# Absorbing Markov Chain Journey Model — Interactive Demo

This notebook provides a step-by-step walkthrough of the absorbing Markov chain
journey model from **Section 5.1** of:

> *Behavioral Intelligence Platforms: From Event Streams to Autonomous Insight*  
> Patra & Vadgave, 2026

---

## Overview

The Behavioral Graph Engine (BGE) models a user journey as an **absorbing Markov chain**
$M_\delta = (\mathcal{S}, \mathcal{A}, Q, R)$ where:

| Symbol | Meaning |
|--------|---------|
| $\mathcal{S}$ | Set of transient (non-terminal) states |
| $\mathcal{A}$ | Set of absorbing (terminal) states |
| $Q$ | Sub-stochastic matrix of transition probabilities among transient states |
| $R$ | Transition probabilities from transient states to absorbing states |

The **fundamental matrix** $N = (I - Q)^{-1}$ gives the expected number of visits
to each transient state before absorption, and the **absorption probability matrix**
$B = N \cdot R$ gives conversion probabilities from each state.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

np.set_printoptions(precision=4, suppress=True)

## 1. Define the journey graph

We model a simple SaaS signup-to-activation funnel:

```
sign_up → email_verified → profile_complete → feature_used → converted
                                                            ↘ dropped
```

In [ ]:
TRANSIENT = ["sign_up", "email_verified", "profile_complete", "feature_used"]
ABSORBING = ["converted", "dropped"]

# Full transition matrix P  (transient states × all states)
P_full = np.array([
    # from sign_up
    [0.00, 0.70, 0.10, 0.00,  0.02, 0.18],
    # from email_verified
    [0.00, 0.00, 0.65, 0.10,  0.05, 0.20],
    # from profile_complete
    [0.00, 0.00, 0.00, 0.75,  0.05, 0.20],
    # from feature_used
    [0.00, 0.00, 0.05, 0.10,  0.60, 0.25],
])

n_t = len(TRANSIENT)
n_a = len(ABSORBING)

print("Row sums (must all equal 1.0):", P_full.sum(axis=1))

## 2. Extract Q and R

In [ ]:
Q = P_full[:, :n_t]   # transient → transient
R = P_full[:, n_t:]   # transient → absorbing

print("Q (transient → transient):")
print(Q)
print("\nR (transient → absorbing):")
print(R)

## 3. Fundamental matrix  $N = (I - Q)^{-1}$

$N_{ij}$ = expected number of times the chain visits transient state $j$
when starting from transient state $i$, before absorption.

In [ ]:
N = np.linalg.inv(np.eye(n_t) - Q)

import pandas as pd
pd.DataFrame(N, index=TRANSIENT, columns=TRANSIENT).round(4)

## 4. Expected remaining steps  $t_i = \sum_j N_{ij}$

In [ ]:
t_vec = N.sum(axis=1)

fig, ax = plt.subplots(figsize=(7, 3))
bars = ax.barh(TRANSIENT[::-1], t_vec[::-1], color="steelblue")
ax.bar_label(bars, fmt="%.2f", padding=3)
ax.set_xlabel("Expected remaining steps until absorption")
ax.set_title("Journey friction by state ($t_i = \\sum_j N_{ij}$)")
plt.tight_layout()
plt.show()

## 5. Absorption probability matrix  $B = N \cdot R$

$B[i, \text{converted}]$ = outcome conversion probability starting from state $i$.

In [ ]:
B = N @ R

pd.DataFrame(B, index=TRANSIENT, columns=ABSORBING).round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
x = np.arange(n_t)
width = 0.35
ax.bar(x - width/2, B[:, 0], width, label="P(converted)", color="#2ecc71")
ax.bar(x + width/2, B[:, 1], width, label="P(dropped)",   color="#e74c3c")
ax.set_xticks(x)
ax.set_xticklabels(TRANSIENT, rotation=15, ha="right")
ax.set_ylabel("Absorption probability")
ax.set_title("Conversion vs drop-off probability from each transient state")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Key insight

States with high conversion probability (`B[:, 0]`) are candidate **activation drivers** 
(Section 5.3). States where a user spends many expected steps (`t_i`) before absorption 
indicate **journey friction**.

In [ ]:
best = np.argmax(B[:, 0])
worst = np.argmin(B[:, 0])
print(f"Highest conversion: '{TRANSIENT[best]}'  → {B[best, 0]:.1%}")
print(f"Lowest  conversion: '{TRANSIENT[worst]}' → {B[worst, 0]:.1%}")
print(f"\nMost friction (highest expected steps): '{TRANSIENT[np.argmax(t_vec)]}'  "
      f"→ {t_vec[np.argmax(t_vec)]:.2f} expected steps")